In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df =pd.read_csv(r"/content/drive/MyDrive/dataset/Superstoredata.csv",encoding='latin1', engine='python')
df


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


DATA CLEANING

In [ ]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)


In [ ]:
# Convert date columns
df['order_date'] = pd.to_datetime(
    df['order_date'],
    errors='coerce'
)

df['ship_date'] = pd.to_datetime(
    df['ship_date'],
    errors='coerce'
)


In [ ]:

# Remove null dates
df = df.dropna(subset=['order_date'])

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Convert numeric columns
df['sales'] = pd.to_numeric(df['sales'])
df['profit'] = pd.to_numeric(df['profit'])

FINANCE KPI CREATION


In [ ]:

# Total Income
df['total_income'] = df['sales']


In [ ]:
# Total Expense
df['total_expense'] = (
    df['sales'] - df['profit']
)

In [ ]:
# Savings
df['savings'] = (
    df['total_income'] - df['total_expense']
)

In [ ]:
# Budget
df['budget'] = (
    df['total_expense'] * 1.10
)

In [ ]:
# Budget Variance
df['budget_variance'] = (
    df['budget'] - df['total_expense']
)


In [ ]:
# Budget Utilization %
df['budget_utilization_%'] = (
    df['total_expense'] /
    (df['total_expense'] * 1.10)
) * 100

# EXPLORATORY DATA ANALYSIS



 MAIN BUSINESS KPIs

In [ ]:
total_income = df['total_income'].sum()
total_expense = df['total_expense'].sum()
total_savings = df['savings'].sum()

print(f"Total Income: ₹{total_income:,.0f}")
print(f"Total Expense: ₹{total_expense:,.0f}")
print(f"Total Savings: ₹{total_savings:,.0f}")


Total Income: ₹2,297,201
Total Expense: ₹2,010,804
Total Savings: ₹286,397


# CATEGORY ANALYSIS

In [ ]:
print(
    df.groupby('category')['total_expense']
    .sum()
    .sort_values(ascending=False)
)


category
Furniture          723548.5225
Technology         690699.0849
Office Supplies    596556.2312
Name: total_expense, dtype: float64


REGION ANALYSIS

In [ ]:
print(
    df.groupby('region')['total_income']
    .sum()
    .sort_values(ascending=False)
)

region
West       725457.8245
East       678781.2400
Central    501239.8908
South      391721.9050
Name: total_income, dtype: float64


TIME-BASED ANALYSIS

In [ ]:
# Create Month and Year columns
df['month'] = df['order_date'].dt.strftime('%b')
df['year'] = df['order_date'].dt.year

# Monthly Financial Summary
monthly = df.groupby('month')[

    [
        'total_income',
        'total_expense',
        'savings'
    ]

].sum().reset_index()

print(monthly)

   month  total_income  total_expense     savings
0    Apr   137762.1286    126174.6923  11587.4363
1    Aug   159044.0630    137267.1246  21776.9384
2    Dec   325293.5035    281924.3116  43369.1919
3    Feb    59751.2514     49456.6407  10294.6107
4    Jan    94924.8356     85790.3895   9134.4461
5    Jul   147238.0970    133405.4322  13832.6648
6    Jun   152718.6793    131432.8839  21285.7954
7    Mar   205005.4888    176410.8016  28594.6872
8    May   155028.8117    132617.5039  22411.3078
9    Nov   352461.0710    316992.6445  35468.4265
10   Oct   200322.9847    168538.9434  31784.0413
11   Sep   307649.9457    270792.4704  36857.4753


In [ ]:
# =========================================================================
# EXPORT FOR POWER BI
# =========================================================================

with pd.ExcelWriter("Financial_Expense_Tracker_Dashboard.xlsx") as writer:

    df.to_excel(
        writer,
        sheet_name="Clean Data",
        index=False
    )

print("Finance Project Export Completed!")

Finance Project Export Completed!


In [ ]:
import os

os.listdir()

['.config',
 'drive',
 'Financial_Expense_Tracker_Dashboard.xlsx',
 'finance_project_cleaned.xlsx',
 'sample_data']

In [ ]:
from google.colab import files

files.download(
    "Financial_Expense_Tracker_Dashboard.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>